# Model 3: Bidirectional LSTM

Recurrent neural network approach to IMDB sentiment classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import re, collections
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── Load & Clean Data ──────────────────────────────────────────────────────
df = pd.read_csv('IMDB Dataset.csv')
df['label'] = (df['sentiment'] == 'positive').astype(int)

def clean(text):
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower().split()

df['tokens'] = df['review'].apply(clean)
print('Sample tokens:', df['tokens'][0][:10])

In [ ]:
# ── Build Vocabulary ───────────────────────────────────────────────────────
MAX_VOCAB = 20000
MAX_LEN   = 256

counter = collections.Counter(t for tokens in df['tokens'] for t in tokens)
vocab   = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
w2i     = {w: i for i, w in enumerate(vocab)}
PAD_IDX, UNK_IDX = 0, 1

def encode(tokens):
    ids = [w2i.get(t, UNK_IDX) for t in tokens[:MAX_LEN]]
    return ids

df['ids'] = df['tokens'].apply(encode)
print(f'Vocab size: {len(vocab)}')

In [ ]:
# ── Dataset & DataLoader ───────────────────────────────────────────────────
class IMDBDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = [torch.LongTensor(x) for x in ids]
        self.labels = torch.LongTensor(labels)
    def __len__(self):  return len(self.labels)
    def __getitem__(self, i): return self.ids[i], self.labels[i]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    seqs_padded  = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    return seqs_padded, torch.stack(labels)

train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=42, stratify=df['label'])
train_idx, val_idx  = train_test_split(train_idx, test_size=0.1, random_state=42)

def make_loader(idx, shuffle=True):
    ds = IMDBDataset(df['ids'].iloc[idx].tolist(), df['label'].iloc[idx].tolist())
    return DataLoader(ds, batch_size=64, shuffle=shuffle, collate_fn=collate_fn)

train_loader = make_loader(train_idx)
val_loader   = make_loader(val_idx,   shuffle=False)
test_loader  = make_loader(test_idx,  shuffle=False)

In [ ]:
# ── Bidirectional LSTM Model ───────────────────────────────────────────────
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 n_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, 2)  # *2 for bidirectional

    def forward(self, x):
        emb = self.dropout(self.embedding(x))        # (B, T, E)
        out, (h, _) = self.lstm(emb)                 # h: (2*L, B, H)
        # Concatenate last forward & backward hidden states
        h = torch.cat([h[-2], h[-1]], dim=1)         # (B, 2H)
        return self.fc(self.dropout(h))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BiLSTM(len(vocab)).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); 
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val, best_state, patience_cnt, PATIENCE = 0, None, 0, 5

for epoch in range(1, 21):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    if va > best_val:
        best_val = va
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}'); break
    print(f'Epoch {epoch:2d} | Train Loss {tl:.4f} Acc {ta:.4f} | Val Loss {vl:.4f} Acc {va:.4f}')

model.load_state_dict(best_state)

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(yb.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {test_acc:.4f}')
print(classification_report(all_labels, all_preds, target_names=['Negative','Positive']))

In [ ]:
# ── Plots & Save ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('BiLSTM Loss'); axes[0].legend()
axes[1].plot(history['train_acc'],  label='Train'); axes[1].plot(history['val_acc'],  label='Val')
axes[1].set_title('BiLSTM Accuracy'); axes[1].legend()
plt.tight_layout(); plt.savefig('../results/lstm_curves.png', dpi=150); plt.show()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
plt.title('Confusion Matrix – BiLSTM')
plt.tight_layout(); plt.savefig('../results/lstm_confusion_matrix.png', dpi=150); plt.show()

report = classification_report(all_labels, all_preds, target_names=['Negative','Positive'])
with open('../results/lstm_results.txt', 'w') as f:
    f.write('=== Bidirectional LSTM Results ===\n\n')
    f.write(f'Test Accuracy: {test_acc:.4f}\n\n')
    f.write('Classification Report:\n'); f.write(report)
    f.write(f'\nBest Val Accuracy: {best_val:.4f}\n')
print('Results saved.')